# Prior-Art Search using RL
## Agentic Search | Multi-Turn | Tool-use  

The objective is to create an Agent to perform prior-art search over patent corpora. mainly The Harvard USPTO Patent Dataset.



In [1]:
import os
import json
import chromadb
from chromadb.api.types import Embeddable, EmbeddingFunction
from chromadb.utils import embedding_functions
import asyncio
from dotenv import load_dotenv
import pandas as pd

load_dotenv()

# Required for Weights & Biases
os.environ["WANDB_API_KEY"] = os.getenv("WANDB_API_KEY")


In [4]:
# Fields to extract from raw JSON data
RELEVANT_FIELDS = [
    # Identifiers & Linking
    "publication_number",
    "application_number",
    "patent_number",
    # Dates (as epoch ints)
    "date_published",
    "filing_date",
    "patent_issue_date",
    "abandon_date",
    # Status & Classes
    "decision",
    "main_cpc_label",
    "main_ipcr_label",
    # Retrievable Text
    "title",
    "abstract",
    "claims",  ## The legally enforceable boundaries of the invention — the essence of what’s protected.
    # "summary",
]

def get_IP_data():
    """Load and filter IP data from JSON files, skipping files with decode errors."""
    ip_files = []
    for file in os.listdir("Patent_data"):
        file_path = os.path.join("Patent_data", file)
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)
                filtered = {key: value for key, value in data.items() if key in RELEVANT_FIELDS}
                ip_files.append(filtered)
        except (UnicodeDecodeError, json.JSONDecodeError) as e:
            print(f"Skipping {file}: {e}")
    return ip_files

patent_data = get_IP_data()

Skipping .DS_Store: 'utf-8' codec can't decode byte 0x80 in position 3131: invalid start byte


#### Designing the multi-turn environment

An episode is “one prior-art search task”.


Initial observation is something like:

You are a patent prior-art search assistant.
Here is a new invention description:
<DESCRIPTION>
You have access to tools:

patent_search(query) – searches in a database of patents (title+abstract).

patent_lookup(patent_id) – shows full details of one patent.
Use the tools as needed, then produce a final ranked list of relevant prior-art patents.


The environment passes this as the initial prompt.

After each tool call, the environment appends the tool output to the conversation and gives that back to the model.

3.2. Actions

    Action = “next model output”. You constrain the format so the model can:

    Call tools:
    e.g. CALL_TOOL patent_search: "solar panel with transparent coating for windows"

    Or produce final answer:
    e.g. FINAL_ANSWER: [US1234..., EP5678..., ...] plus explanation.

The environment will:

    Parse the model’s output.

    If it’s CALL_TOOL, execute the underlying Python function and append result.

    If it’s FINAL_ANSWER, end the episode and compute reward.

This is exactly the pattern used in TRL’s experimental “learning tools” examples with calculator/wiki/python environments. 


3.3. Termination

Terminate when:

    The model outputs FINAL_ANSWER, or

    You hit max_steps (e.g. 5–8 tool calls) and then force the model to give a final answer, or mark as failure.

In [2]:
CHROMA_DB_DIR = ".chroma_db"
_chroma_semaphore: asyncio.Semaphore | None = None

def get_chroma_semaphore() -> asyncio.Semaphore:
    global _chroma_semaphore
    if _chroma_semaphore is None:
        _chroma_semaphore = asyncio.Semaphore(20)
    return _chroma_semaphore


def init_chroma_collection(force_recreate: bool = False) :
    """Connect to an existing ChromaDB collection for patent data, or create if not exists."""
    chroma_client = chromadb.PersistentClient(path=CHROMA_DB_DIR)
    # get the collection if it exists
    try:
        if force_recreate:
            chroma_client.delete_collection(name="patent_collection")
        collection = chroma_client.get_collection(
            name="patent_collection",
            embedding_function=embedding_functions.SentenceTransformerEmbeddingFunction(
                model_name="sentence-transformers/all-mpnet-base-v2"
            )
        )
        return collection
    except Exception:
        # If not found, create and ingest
        collection = chroma_client.get_or_create_collection(
            name="patent_collection",
            embedding_function=embedding_functions.SentenceTransformerEmbeddingFunction(
                model_name="sentence-transformers/all-mpnet-base-v2"
            ),
            configuration={
                "hnsw": {
                    "space": "cosine",
                    "ef_construction": 200,
                    "ef_search": 150
                }
            }
        )
        collection.add(
            documents=[patent["abstract"] for patent in patent_data],  # Using abstract as the main text for embeddings
            ids=[patent["publication_number"] for patent in patent_data],
            metadatas=[{k: v for k, v in patent.items() if k != 'abstract'} for patent in patent_data],
        )
        return collection

collection = init_chroma_collection(force_recreate=False)



In [5]:
collection.get("US20180137422A1-20180517")

{'ids': ['US20180137422A1-20180517'],
 'embeddings': None,
 'documents': ['Methods of training Boltzmann machines include rejection sampling to approximate a Gibbs distribution associated with layers of the Boltzmann machine. Accepted sample values obtained using a set of training vectors and a set of model values associate with a model distribution are processed to obtain gradients of an objective function so that the Boltzmann machine specification can be updated. In other examples, a Gibbs distribution is estimated or a quantum circuit is specified so at to produce eigenphases of a unitary.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'publication_number': 'US20180137422A1-20180517',
   'patent_issue_date': '',
   'application_number': '15579190',
   'main_cpc_label': 'G06N5022',
   'main_ipcr_label': 'G06N502',
   'abandon_date': '',
   'patent_number': 'None',
   'date_published': '20180517',
   'decision': 'PENDING',
   'claims': '1.-1

### Defining The Tools

In [6]:
from pydantic import BaseModel, Field
from textwrap import dedent
from dataclasses import dataclass, field, asdict
from typing import Any, Dict, List, Optional, get_origin, get_args


## search tool
async def search_patents(query: str, n_results: int = 10) -> list[dict]:
    """Search for top 10 relevant patents using title embedding similarity."""
    async with get_chroma_semaphore():
        results = collection.query(
            query_texts=[query],
            n_results=n_results
        )
    
    if not results:
        raise ValueError(f"No results found for query: {query}")
    if not results["metadatas"]:
        raise ValueError(f"No results metadata found for query: {query}")
    
    output = []
    for i in range(len(results["ids"][0])):
        patent_title = results["metadatas"][0][i]["title"]
        publication_number = results["ids"][0][i]
        similarity_score = results["distances"][0][i]
        output.append({"patent_title": patent_title, 
                       "publication_number": publication_number,
                       "similarity_score": similarity_score})
    
    return output

# Patent lookup tool
async def lookup_patent(publication_number: str) -> dict:
    """Lookup patent details by publication number."""
    sem = get_chroma_semaphore()
    async with sem:
        results = await asyncio.to_thread(
            collection.get,
            ids=[publication_number],
        )

    if not results or not results.get("metadatas"):
        raise ValueError(f"No patent found with publication number: {publication_number}")

    patent_content = results["documents"][0]
    patent_metadata = results["metadatas"][0]
    return {**patent_metadata, "abstract": patent_content}

collection = init_chroma_collection(force_recreate=False)


In [7]:
res = await search_patents("battery management", n_results=3)
res

[{'patent_title': 'QUICK-EXCHANGE BATTERY ASSEMBLY, AND MOTOR VEHICLE, IN PARTICULAR MOTOR SCOOTER',
  'publication_number': 'US20180151860A1-20180531',
  'similarity_score': 0.5740572214126587},
 {'patent_title': 'METHOD FOR CALCULATING A SETPOINT FOR MANAGING THE FUEL AND ELECTRICITY CONSUMPTION OF A HYBRID MOTOR VEHICLE',
  'publication_number': 'US20180281620A1-20181004',
  'similarity_score': 0.6117575168609619},
 {'patent_title': 'POSITIVE ELECTRODE ACTIVE MATERIAL FOR RECHARGABLE LITHIUM BATTERY, METHOD FOR MANUFACTURING SAME, AND RECHARGABLE LITHIUM BATTERY INCLUDING SAME',
  'publication_number': 'US20180254473A1-20180906',
  'similarity_score': 0.6518891453742981}]

## Reward Design for prior-art Search

### Backward-citation

We need labels for what good prior art looks like.
since we dont have 'citations' in the documents


the ground truth will be basically patents abstract snippets and their backward citations as ground truth.
For a given patent P, the cited patents = “true” prior art for P.


In [8]:
import dspy
from dotenv import load_dotenv
import os
import pandas as pd

load_dotenv()
base_url = os.getenv("GEMINI_API_BASE", "https://generativelanguage.googleapis.com/v1beta")
api_key = os.getenv("GEMINI_API_KEY")
lm = dspy.LM('gemini/gemini-2.0-flash', api_key=api_key)
dspy.configure(lm=lm)

class ExtractInfo(dspy.Signature):
    """you are gonna be given an abtract of a patent and you need to generate 3 queries that
    can be used to search for prior art patents related to the given patent abstract. 
    The queries should be concise and relevant to the key aspects of the patent abstract.
    only return the queries as a json array of strings with no other text."""

    patent_abstract: str = dspy.InputField()
    number_of_queries: int = dspy.InputField(default=3, desc="Number of search queries to generate.")
    queries: list[str] = dspy.OutputField(desc="List of search queries for prior art patents.")

module = dspy.Predict(ExtractInfo)

def get_queries_from_abstract(patent_abstract: str, number_of_queries=3) -> list[str]:
    """Generate search queries for prior art patents based on the given patent abstract."""
    try:
        return module(patent_abstract=patent_abstract, 
                      number_of_queries=number_of_queries).queries
    except Exception as e:
        raise print(f"Error generating queries: {e}")
        
def create_search_queries(patent_data: list[str], number_of_queries=3)-> pd.DataFrame:
    """Generate search queries for a list of patent abstracts."""
    all_queries = []
    for patent in patent_data:
        queries = get_queries_from_abstract(patent['abstract'], number_of_queries)
        for query in queries:
            all_queries.append({'publication_number': patent['publication_number'], 'query': query, 'abstract': patent['abstract']})
    out = pd.DataFrame(all_queries)
    out.to_csv("Evals/patent_search_queries.csv", index=False)
    return out
# df = create_search_queries(patent_data, number_of_queries=3)

In [8]:
patent_search_queries = pd.read_csv("Evals/patent_search_queries.csv")
patent_search_queries.iloc[0].to_dict()

{'publication_number': 'US20180271722A1-20180927',
 'query': 'disposable absorbent article vital signs monitoring system',
 'abstract': 'The present invention protects a system for monitoring the vital signs of a user of a disposable absorbent article that comprises a sensor device, a bi-dimensional code and a receiver, such that the sensor device is placed in contact with the user and the system is triggered by the reading of the bi-dimensional code through the receiver, such that the bi-dimensional code is place inside the packaging or the disposable absorbent article.'}

In [46]:
import json
import logging
import inspect
from dataclasses import dataclass
from textwrap import dedent
from typing import Any, Dict, List, Optional

from langchain_core.utils.function_calling import convert_to_openai_tool
from openai import AsyncOpenAI
from pydantic import BaseModel

import art


MAX_TURNS = 6


# Patent data models

class Patent(BaseModel):
    publication_number: str
    title: str
    abstact: Optional[str] = None
    main_ipcr_label: Optional[str] = None
    main_cpc_label: List[str] = []
    decision: List[str] = []
    patent_issue_date: List[str] = []


@dataclass
class SearchResult:
    message_id: str
    snippet: str


class FinalAnswer(BaseModel):
    answer: str
    patent_ids: List[str]


class ProjectTrajectory(art.Trajectory):
    # Reuse the Trajectory type from `art`, only
    # adding the final_answer field specific to this project.
    final_answer: Optional[FinalAnswer] = None


@dataclass
class JudgeResponse:
    accept: bool


def return_final_answer(answer: str, patent_ids: List[str]) -> FinalAnswer:
    """Return the final answer and the patent IDs that support it."""
    return FinalAnswer(answer=answer, patent_ids=patent_ids)


async def judge_correctness(scenario_row: Dict[str, Any], final_answer: FinalAnswer) -> JudgeResponse:
    gold = str(scenario_row["publication_number"])
    accept = gold in final_answer.patent_ids
    return JudgeResponse(accept=accept)


# Multi‑turn rollout for prior‑art search
async def rollout(model: art.Model, search_scenario: Dict[str, Any]) -> ProjectTrajectory:
    scenario = search_scenario

    traj = ProjectTrajectory(
        reward=0.0,
        messages_and_choices=[],
        metadata={
            "query": scenario.get("query")},
    )

    system_prompt = dedent(
        f"""
        You are a prior-art search agent. You are given a new invention description
        or search query and a set of tools you can use to perform prior-art search.

        Use the tools to search for relevant prior patents in the patent database
        and identify the most relevant patent publication numbers.

        You may take up to {MAX_TURNS} turns; if your first search does not find
        the answer, refine your queries and try again.
        You should only return a maximum of 3 publication numbers of patents that are relevant
        to the invention description or query provided.

        Tools:
        1. search_patents(query: str, n_results: int = 10) -> list[dict]:
           Search for top n_results relevant patents using embedding similarity.
        2. lookup_patent(publication_number: str) -> dict:
           Lookup patent details by publication number.
        3. return_final_answer(answer: str, patent_ids: list[str]) -> FinalAnswer:
           Return the final answer and the list of patent publication numbers
           that support your answer.
        """
    )

    user_prompt_parts = [f"New invention description or query:\n{scenario['query']}"]
    user_prompt = "\n".join(user_prompt_parts)

    traj.messages_and_choices = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    tools = [search_patents, lookup_patent, return_final_answer]
    tools_by_name = {t.__name__: t for t in tools}

    # Use the same tool representation as in the `art` examples.
    traj.tools = [convert_to_openai_tool(t) for t in tools]

    client = AsyncOpenAI(
        base_url=model.inference_base_url,
        api_key=model.inference_api_key,
    )

    for _ in range(MAX_TURNS):
        response = await client.chat.completions.create(
            model=model.get_inference_name(),
            temperature=1.0,
            messages=traj.messages(),
            tools=traj.tools,
        )

        response_message = response.choices[0].message
        traj.messages_and_choices.append(response.choices[0])

        # No tool calls → end episode
        if not response_message.tool_calls:
            return traj

        try:
            for tool_call in response_message.tool_calls:
                tool_name: str = tool_call.function.name
                if tool_name not in tools_by_name:
                    continue

                tool_args = json.loads(tool_call.function.arguments or "{}")
                tool_to_call = tools_by_name[tool_name]

                # Support async and sync tools
                if inspect.iscoroutinefunction(tool_to_call):
                    result = await tool_to_call(**tool_args)
                else:
                    result = tool_to_call(**tool_args)

                traj.messages_and_choices.append(
                    {
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "name": tool_name,
                        "content": json.dumps(result, default=str),
                    }
                )

                if tool_name == "return_final_answer":
                    traj.final_answer = result

                    if traj.final_answer:
                        correctness_judge_response = await judge_correctness(
                            scenario, traj.final_answer
                        )
                        traj.metrics["correct"] = float(
                            correctness_judge_response.accept
                        )
                    return traj

        except Exception as e:
            logging.error(f"Error executing tool call: {e}")
            return traj

    return traj


### Training

In [47]:
import art
from art.serverless.backend import ServerlessBackend

model = art.TrainableModel(
  project="Patent-Search",
  name="agent-001",
  base_model="OpenPipe/Qwen3-14B-Instruct"
)

# Initialize the server
# Training and inference will run on Weights & Biases servers
backend = ServerlessBackend()

# Register the model with the Serverless Backend (sets up logging, inference, and training)
await model.register(backend)

In [52]:
res = await rollout(model, train_dataset.iloc[3])

In [54]:
res.messages_and_choices

[{'role': 'system',
  'content': '\nYou are a prior-art search agent. You are given a new invention description\nor search query and a set of tools you can use to perform prior-art search.\n\nUse the tools to search for relevant prior patents in the patent database\nand identify the most relevant patent publication numbers.\n\nYou may take up to 6 turns; if your first search does not find\nthe answer, refine your queries and try again.\nYou should only return a maximum of 3 publication numbers of patents that are relevant\nto the invention description or query provided.\n\nTools:\n1. search_patents(query: str, n_results: int = 10) -> list[dict]:\n   Search for top n_results relevant patents using embedding similarity.\n2. lookup_patent(publication_number: str) -> dict:\n   Lookup patent details by publication number.\n3. return_final_answer(answer: str, patent_ids: list[str]) -> FinalAnswer:\n   Return the final answer and the list of patent publication numbers\n   that support your an

In [50]:
from prior_art_search.training_loop_qwen import get_train_val_sets
train_dataset, val_dataset = get_train_val_sets()
train_dataset

,publication_number,query,abstract
307,US20180179041A1-20180628,beverage filling plant container transport hol...,"A holding device for holding a container, for ..."
1089,US20180126484A1-20180510,laser induced detachment zone solid,The invention relates to a method for creating...
741,US20180175922A1-20180621,multi-antenna channel estimation control,A multi-antenna apparatus for controlling tran...
237,US20180133333A1-20180517,thermo-responsive polymer lower critical solut...,The invention provides for novel thermo-respon...
952,US20180239949A1-20180823,automated cell analysis system,"Methods, systems, and devices are provided for..."
...,...,...,...
1782,US20180105514A1-20180419,compounds formula q r s A B C RA1 RA2 RB1 RB2 ...,Disclosed are compounds having the formula: wh...
1951,US20180199657A1-20180719,shoe sensor data transmission output control,Adaptive output control is performed on the ba...
1746,US20180265282A1-20180920,foldable container hinged side plates curved s...,"A foldable container, comprising a base and tw..."
1720,US20180249976A1-20180906,radiation dose electric charge accumulation pi...,Improvement of a frame rate and suppression of...


#### sanity-check reward function

In [51]:
import art
from art.rewards import ruler_score_group

# Test RULER with a simple example
base_messages = [
    {"role": "system", "content": "you return a list of patent publication numbers relevant to the query"},
    {"role": "user", "content": "Count to 10."},
]

good_trajectory = art.Trajectory(
    messages_and_choices=[
        *base_messages,
        {"role": "assistant", 
         "content": [US20180249976A1-20180906]},
    ],
    reward=0,
)

mediocre_trajectory = art.Trajectory(
    messages_and_choices=[
        *base_messages,
        {
            "role": "assistant",
            "content": "one, two, three, four, five, six, seven, eight, nine, ten",
        },
    ],
    reward=0,
)

bad_trajectory = art.Trajectory(
    messages_and_choices=[
        *base_messages,
        {"role": "assistant", "content": "a, b, c, d, e, f, g, h, i, j"},
    ],
    reward=0,
)

sample_group = art.TrajectoryGroup(
    trajectories=[
        good_trajectory,
        mediocre_trajectory,
        bad_trajectory,
    ]
)

judged_group = await ruler_score_group(sample_group, , debug=True)
assert judged_group is not None

# Display rankings
sorted_trajectories = sorted(
    judged_group.trajectories, key=lambda t: t.reward, reverse=True
)
for rank, traj in enumerate(sorted_trajectories, 1):
    messages = traj.messages()
    print(f"\nRank {rank}: Score {traj.reward:.3f}")
    print(f"  Response: {messages[-1]['content'][:50]}...")

SyntaxError: invalid syntax (240189691.py, line 46)

### RULER:  LLM-as-a-judge
As in RULER from openpipe


RULER (Relative Universal LLM-Elicited Rewards) is a general-purpose reward function that uses an LLM-as-judge to rank multiple agent trajectories. 

It requires no labeled data, expert feedback, or hand-crafted reward functions, yet reliably improves agent performance.